# 4.12 · 随机森林回归 / Random Forest Regressor

> **课程定位 / Where this fits**
> **Part 4 第 12 课**。4.11 的单棵树**高方差**。随机森林 = **训练很多棵去相关的树取平均** → 方差骤降, 偏差几乎不变。它**几乎零调参、不需缩放、自带特征重要性和 OOB 验证**, 是表格数据的"安全默认"。本课讲清 bagging 数学和"去相关"的两层随机。
> Many decorrelated trees averaged → variance crashes, bias stays. Near-zero-tuning, no-scaling, with built-in importance and OOB validation.

> 💡 **面试相关 / Interview-relevant**
> - "随机森林为什么比单棵树好" ★★★★★（方差缩减数学）
> - "bagging 是什么" ★★★★
> - "随机森林的两层随机" ★★★★（bootstrap 行 + 特征子集列）
> - "OOB 是什么, 怎么用" ★★★★（免费验证）
> - "RF vs GBDT" ★★★★

---

## 学习目标 / Learning Objectives
1. 理解 **bagging** 的方差缩减数学：平均 $B$ 棵树, 方差降至 $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$。
2. 理解**两层随机**（bootstrap 行 + 每次切分随机特征子集）如何**去相关**。
3. 用 **OOB** 免费估计泛化误差（无需单独验证集）。
4. 树数 / max_features 等超参的作用（几乎不会过拟合于树数）。

## 目录 / TOC
1. [bagging: 平均缩减方差 ⭐](#1)
2. [两层随机: 去相关 ⭐](#2)
3. [💎 数据 + 树数效应](#3)
4. [OOB: 免费验证 ⭐](#4)
5. [超参 + 特征重要性](#5)
6. [RF vs 单树 vs GBDT 预告](#6)
7. [小结](#7)


<a id="1"></a>
## 1. bagging: 平均缩减方差 ⭐ / Bagging Reduces Variance

**核心数学**（接 0.10 偏差-方差、2.4 bootstrap）。$B$ 个相关系数为 $\rho$、各自方差 $\sigma^2$ 的估计器, 平均后的方差：

$$\text{Var}\Big(\frac{1}{B}\sum_b T_b\Big) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

**两层洞察**:
1. **第二项随 $B$ 减小** → 树越多方差越低（但有下限）
2. **第一项 $\rho\sigma^2$ 是下限** → 即使无限棵树, 方差也降不到 0, **除非让树之间去相关（$\rho$ 小）**

→ 这就是随机森林比简单 bagging（只 bootstrap）更强的原因: 它**额外引入特征随机来降低 $\rho$**（第 2 节）。

**Bagging (Bootstrap Aggregating)**: 每棵树用一个 **bootstrap 样本**（2.4: 有放回抽 n 个）训练 → 树之间天然有差异 → 平均降方差。**偏差几乎不变**（每棵树仍是无偏的, 平均还是无偏）, 但方差骤降。
Averaging B trees: variance = rho*sigma^2 + (1-rho)/B*sigma^2. More trees shrink the second term; decorrelation shrinks the first floor.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

df = sns.load_dataset("diamonds").sample(8000, random_state=0).reset_index(drop=True)
X = df[["carat","depth","table","x","y","z"]].values
y = df["price"].values
feat_names = ["carat","depth","table","x","y","z"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
print(f"Diamonds: {X.shape}")


In [ ]:
# 演示方差缩减: 单树 vs 森林的预测稳定性 / variance reduction demo
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# 多次 bootstrap 训练, 看预测波动 / fit on bootstrap samples, see prediction spread
x_test_pt = X_te[:1]                       # 固定一个测试点
single_preds, forest_preds = [], []
for s in range(50):
    idx = rng.choice(len(X_tr), len(X_tr), replace=True)
    single_preds.append(DecisionTreeRegressor(max_depth=10, random_state=s).fit(X_tr[idx], y_tr[idx]).predict(x_test_pt)[0])
    forest_preds.append(RandomForestRegressor(n_estimators=30, max_depth=10, random_state=s, n_jobs=-1).fit(X_tr[idx], y_tr[idx]).predict(x_test_pt)[0])

print(f"对同一测试点, 50 次重训的预测标准差:")
print(f"  单棵树:   std = {np.std(single_preds):.1f}  (高方差, 预测很不稳)")
print(f"  随机森林: std = {np.std(forest_preds):.1f}  (方差骤降, 预测稳定)")
print(f"→ 森林预测稳定 {np.std(single_preds)/np.std(forest_preds):.1f} 倍 — bagging 方差缩减的直接证据")


<a id="2"></a>
## 2. 两层随机: 去相关 ⭐ / Two Sources of Randomness

随机森林 = bagging **+ 特征随机**, 两层随机让树**去相关**（降低公式里的 $\rho$）：

| 随机来源 | 做法 | 作用 |
|---|---|---|
| **1. Bootstrap 行** | 每棵树用有放回抽样的训练集 | 树看到不同样本 |
| **2. 特征子集 (列)** ⭐ | **每次切分**只考虑随机的 `max_features` 个特征 | 防止所有树都被同一强特征主导 → **去相关** |

**为什么第 2 层关键**: 若只 bootstrap（普通 bagging）, 每棵树都会优先用最强特征（carat）切第一刀 → 树高度相似（$\rho$ 大）→ 平均效果有限。**特征随机强迫树用不同特征** → 树各异 → $\rho$ 小 → 方差降更多。

回归默认 `max_features=1.0`(全部, 即纯 bagging) 或 `1/3`(更去相关); 分类默认 $\sqrt{d}$。
The feature subsampling decorrelates trees by preventing the strongest feature from dominating every first split.


<a id="3"></a>
## 3. 树数效应 / Number of Trees


In [ ]:
# 树数 vs 性能: 单调提升后饱和, 几乎不过拟合 / more trees -> better then plateau
n_trees = [1, 5, 10, 25, 50, 100, 200]
test_r2 = []
for nt in n_trees:
    rf = RandomForestRegressor(n_estimators=nt, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    test_r2.append(rf.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_trees, test_r2, "o-", lw=2)
ax.set_xlabel("树数 n_estimators"); ax.set_ylabel("test R²")
ax.set_title("树数 vs 性能: 上升后饱和, 不会因树多而过拟合")
plt.tight_layout(); plt.show()
print(f"{'树数':<8} {'test R²':<10}")
for nt, r in zip(n_trees, test_r2): print(f"{nt:<8} {r:.4f}")
print("\n关键: 树越多越好(单调), 然后饱和 — 随机森林不会因树多而过拟合")
print("(对比 boosting 4.13 会因迭代过多过拟合) → RF 的 n_estimators 越大越安全, 只是变慢")


<a id="4"></a>
## 4. OOB: 免费验证 ⭐ / Out-of-Bag Estimation

**随机森林的独门福利**。2.4 节算过: 每个 bootstrap 样本平均含 ~63.2% 的原始数据 → **每棵树有约 36.8% 的样本没见过**（out-of-bag, OOB）。

**OOB 评估**: 用每个样本"没参与训练的那些树"来预测它 → **免费得到一个验证分数, 无需单独留出验证集**。在数据珍贵时尤其有用。
Each tree leaves out ~36.8% of data; predicting each sample with only the trees that didn't see it gives a free validation score.


In [ ]:
rf_oob = RandomForestRegressor(n_estimators=200, oob_score=True, random_state=0, n_jobs=-1)
rf_oob.fit(X_tr, y_tr)

print(f"OOB R² (免费, 没用 test):  {rf_oob.oob_score_:.4f}")
print(f"真正的 test R²:            {rf_oob.score(X_te, y_te):.4f}")
print(f"5-fold CV R²:             {cross_val_score(RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1), X_tr, y_tr, cv=5).mean():.4f}")
print("\nOOB ≈ test ≈ CV → OOB 是可靠的泛化估计, 且零额外成本(训练时顺便算出)")
print("💡 数据珍贵时, RF 的 OOB 让你不必牺牲数据做验证集")


<a id="5"></a>
## 5. 超参 + 特征重要性 / Hyperparameters & Importance

| 超参 | 作用 | 建议 |
|---|---|---|
| `n_estimators` | 树数 | 越多越好, 看算力(200-500 常够) |
| `max_features` | 每次切分的特征子集 | 回归试 1/3 或 1.0; 调它影响去相关 |
| `max_depth` / `min_samples_leaf` | 单树复杂度 | RF 通常让树长深(靠平均控方差) |
| `n_jobs=-1` | 并行 | **树之间独立 → 完美并行** ⭐ |


In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
print("随机森林特征重要性:")
print(imp.round(3))

fig, ax = plt.subplots(figsize=(7, 3.5))
imp.plot(kind="barh", ax=ax); ax.invert_yaxis()
ax.set_title("RF 特征重要性: carat + 尺寸(x,y,z) 主导钻石价格")
plt.tight_layout(); plt.show()
print("\n⚠ 注意: 默认的 impurity-based 重要性偏向高基数/连续特征")
print("更可靠的是 permutation importance (5.7 节) — 打乱一列看性能掉多少")


<a id="6"></a>
## 6. RF vs 单树 vs GBDT 预告 / Comparison


In [ ]:
from sklearn.preprocessing import StandardScaler

models = {
    "单棵树(depth=10)": DecisionTreeRegressor(max_depth=10, random_state=0),
    "随机森林(200)":     RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
}
print(f"{'模型':<18} {'test R²':>9}")
for name, m in models.items():
    m.fit(X_tr, y_tr)
    print(f"{name:<20} {m.score(X_te, y_te):>9.4f}")
print(f"\n随机森林显著优于单树 (方差缩减); GBDT (4.13) 通常还能再高一截")
print("RF 定位: 强力、稳健、几乎零调参的'安全默认'; 要榨最后几个点用 GBDT/XGBoost")

# 验证不需缩放 / RF needs no scaling either
raw = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1), X_tr, y_tr, cv=3).mean()
sc = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1), StandardScaler().fit_transform(X_tr), y_tr, cv=3).mean()
print(f"\n不缩放={raw:.4f}, 缩放={sc:.4f} → RF 同样不需缩放(继承自树)")


### RF vs GBDT（核心对比, 面试常问）

| | 随机森林 (bagging) | GBDT (boosting) |
|---|---|---|
| 树怎么建 | **并行独立**, 各看 bootstrap | **依次纠错**, 后树修前树残差 |
| 降什么 | **方差**(平均高方差树) | **偏差**(逐步逼近) |
| 单树 | 深(低偏差高方差) | 浅(高偏差低方差) |
| 过拟合 | 树多不会过拟合 | **树多会过拟合**(需 early stopping) |
| 调参 | 少 | 多(lr/depth/n_estimators 交互) |
| 并行 | ✅ 完美 | ❌ 序列(树间有依赖) |
| 精度 | 强 | 通常**更强**(榨干性能) |


<a id="7"></a>
## 7. 小结 / Summary

```
随机森林 = 很多去相关的树取平均 → 方差骤降, 偏差不变
方差公式: ρσ² + (1-ρ)/B·σ²  → 树多降第二项, 去相关降第一项下限
两层随机 ⭐: bootstrap 行(样本) + 特征子集 列(每次切分) → 去相关(降ρ)
树数: 越多越好然后饱和, 不会过拟合 (vs boosting 会)
OOB ⭐: ~36.8% 样本未见 → 免费验证, 无需留出验证集
不需缩放(继承树); 完美并行(树独立); 自带特征重要性
RF vs GBDT: 平均降方差 vs 纠错降偏差; RF 零调参安全, GBDT 精度更高
```

### 💡 面试速查
1. **RF 比单树好**: bagging 平均缩减方差 (ρσ²+(1-ρ)/B·σ²)
2. **两层随机**: bootstrap 行 + 每次切分随机特征列 → 去相关
3. **OOB**: 36.8% 未见样本 = 免费验证
4. **RF 树多不过拟合**(只饱和); GBDT 会 → early stopping
5. **RF=降方差(bagging), GBDT=降偏差(boosting)**

### 下一节
**4.13 梯度提升 GBDT + XGBoost**——RF 并行降方差, GBDT 串行降偏差: 每棵新树拟合前面的**残差**(梯度), 逐步逼近。Kaggle 表格赛冠军模型。
